# 📰 Fake News Detection using NLP
### A complete ML pipeline: preprocessing → TF-IDF → model training → evaluation

## 1. Install & Import Libraries

In [1]:
# Install required libraries (run once)
!pip install pandas numpy scikit-learn matplotlib seaborn nltk wordcloud

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import warnings
warnings.filterwarnings('ignore')

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, PassiveAggressiveClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline

from wordcloud import WordCloud

# Download NLTK data
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

print('✅ All libraries imported successfully!')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/satyamraj/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /Users/satyamraj/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/satyamraj/nltk_data...


✅ All libraries imported successfully!


[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


## 2. Load Dataset
> Download dataset from Kaggle: https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset
> Place `Fake.csv` and `True.csv` in the same folder as this notebook.

In [3]:
# ── Option A: Load from Kaggle CSV files ──────────────────────────────────────
try:
    fake_df = pd.read_csv('Fake.csv')
    true_df = pd.read_csv('True.csv')

    fake_df['label'] = 0   # 0 = Fake
    true_df['label'] = 1   # 1 = Real

    df = pd.concat([fake_df, true_df], ignore_index=True)
    df = df[['title', 'text', 'label']].copy()
    df['content'] = df['title'].fillna('') + ' ' + df['text'].fillna('')
    print(f'✅ Dataset loaded from CSV: {df.shape[0]:,} articles')

except FileNotFoundError:
    # ── Option B: Built-in sample dataset (for demo / testing) ────────────────
    print('⚠️  Fake.csv / True.csv not found — using built-in sample data.')
    print('   For full accuracy, download the Kaggle dataset.')

    sample_data = {
        'content': [
            # FAKE examples
            'SHOCKING: Government hiding alien contact for decades! Anonymous insider reveals truth that mainstream media won't tell you. Share before deleted!',
            'BREAKING: Scientists CONFIRM vaccines cause autism. Big Pharma cover-up exposed. Doctors are lying to you about what goes in your body!',
            'You won't believe what they found in tap water! 5G towers activated to control population. Whistleblower risks life to expose globalist agenda.',
            'Celebrity CAUGHT in massive scandal! Sources say this will bring down entire government. The elites don't want you to know this secret.',
            'Miracle cure hidden by pharmaceutical companies! One simple trick doctors refuse to tell patients. Works 100% guaranteed every time!',
            'Deep state planning to cancel elections, insider reveals. Mainstream media is complicit in the biggest hoax ever perpetrated on citizens.',
            'Bill Gates microchip found in COVID vaccine by independent researcher. This is not a conspiracy, this is FACT. Spread the truth now!',
            'Moon landing was FAKED, new evidence proves it. NASA whistleblower speaks out after 50 years of silence about the great deception.',
            'Drinking bleach cures cancer says anonymous doctor. Medical establishment suppressing natural cures to keep patients sick and paying.',
            'George Soros funds antifa army to destroy America. Secret documents leaked proving globalist takeover plan is already underway.',

            # REAL examples
            'The Federal Reserve raised interest rates by 25 basis points on Wednesday, citing continued concerns about inflation. The decision was unanimous among voting members of the Federal Open Market Committee.',
            'Researchers at Johns Hopkins University published findings in the New England Journal of Medicine showing a new drug reduced tumor size in 60 percent of trial participants.',
            'The United Nations Security Council voted 14-1 on Thursday to extend peacekeeping operations in the region for an additional 12 months, with Russia casting the sole dissenting vote.',
            'Apple reported quarterly earnings of $90.1 billion, slightly above analyst expectations. The company attributed growth to strong iPhone and services revenue in international markets.',
            'Climate scientists at NASA released data showing global average temperatures in 2024 were 1.3 degrees Celsius above pre-industrial levels, making it the warmest year on record.',
            'The Supreme Court ruled 6-3 that the lower court must reconsider its decision on environmental regulations, sending the case back for further review.',
            'A study published in Nature found that Mediterranean diet adherence was associated with a 23 percent lower risk of cardiovascular disease over a 10-year follow-up period.',
            'The European Central Bank announced it would maintain its current bond-buying program through the third quarter, contingent on economic conditions meeting its targets.',
            'Engineers at SpaceX successfully tested the Raptor 3 engine, achieving a record thrust of 280 metric tons, according to a company statement released Monday.',
            'WHO officials confirmed that polio eradication efforts in Pakistan have reduced cases by 97 percent over the last decade, citing improved vaccination coverage.',
        ],
        'label': [0]*10 + [1]*10
    }
    df = pd.DataFrame(sample_data)
    print(f'✅ Sample dataset created: {len(df)} articles')

df.head()

SyntaxError: unterminated string literal (detected at line 22) (3319248095.py, line 22)

## 3. Exploratory Data Analysis (EDA)

In [6]:
print('=== Dataset Info ===')
print(f'Total samples : {len(df):,}')
print(f'Fake news     : {(df.label==0).sum():,} ({(df.label==0).mean()*100:.1f}%)')
print(f'Real news     : {(df.label==1).sum():,} ({(df.label==1).mean()*100:.1f}%)')
print(f'Null values   : {df["content"].isna().sum()}')
df['label'].value_counts()

=== Dataset Info ===


NameError: name 'df' is not defined

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class distribution
counts = df['label'].value_counts()
axes[0].bar(['Fake (0)', 'Real (1)'], counts.values, color=['#ef4444', '#22c55e'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Articles')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + max(counts)*0.01, str(v), ha='center', fontweight='bold')

# Article length distribution
df['text_length'] = df['content'].str.len()
axes[1].hist(df[df.label==0]['text_length'], bins=40, alpha=0.7, color='#ef4444', label='Fake')
axes[1].hist(df[df.label==1]['text_length'], bins=40, alpha=0.7, color='#22c55e', label='Real')
axes[1].set_title('Article Length Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Character Count')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.savefig('eda_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA charts saved')

## 4. Text Preprocessing

In [ ]:
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    """Full NLP preprocessing pipeline."""
    if not isinstance(text, str):
        return ''

    # 1. Lowercase
    text = text.lower()

    # 2. Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # 3. Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # 4. Remove punctuation & special characters
    text = re.sub(r'[^a-z\s]', '', text)

    # 5. Tokenize
    tokens = word_tokenize(text)

    # 6. Remove stopwords & short tokens
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]

    # 7. Stemming
    tokens = [stemmer.stem(t) for t in tokens]

    return ' '.join(tokens)

print('Preprocessing text... (may take a moment on large datasets)')
df['clean_text'] = df['content'].apply(preprocess_text)

print('\nBefore preprocessing:')
print(df['content'].iloc[0][:200])
print('\nAfter preprocessing:')
print(df['clean_text'].iloc[0][:200])
print('\n✅ Preprocessing complete!')

## 5. Word Clouds

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

fake_text = ' '.join(df[df.label == 0]['clean_text'])
real_text = ' '.join(df[df.label == 1]['clean_text'])

for ax, text, title, color in [
    (axes[0], fake_text, '🔴 Fake News — Common Words', 'Reds'),
    (axes[1], real_text, '🟢 Real News — Common Words', 'Greens')
]:
    if text.strip():
        wc = WordCloud(width=700, height=400, background_color='white',
                       colormap=color, max_words=80, collocations=False)
        wc.generate(text)
        ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(title, fontsize=13, fontweight='bold', pad=10)

plt.tight_layout()
plt.savefig('wordclouds.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Word clouds saved')

## 6. Feature Extraction — TF-IDF

In [ ]:
X = df['clean_text']
y = df['label']

# Train/test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# TF-IDF Vectorizer
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),     # Unigrams + bigrams
    sublinear_tf=True,      # Apply log normalization
    min_df=2
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f'Training samples : {X_train_tfidf.shape[0]:,}')
print(f'Test samples     : {X_test_tfidf.shape[0]:,}')
print(f'TF-IDF features  : {X_train_tfidf.shape[1]:,}')

## 7. Train & Compare Multiple Models

In [ ]:
models = {
    'Logistic Regression'         : LogisticRegression(max_iter=1000, C=1.0),
    'Passive Aggressive Classifier': PassiveAggressiveClassifier(max_iter=50),
    'Linear SVM'                  : LinearSVC(max_iter=1000),
    'Multinomial Naive Bayes'      : MultinomialNB(alpha=0.1),
}

results = {}

print(f"{'Model':<35} {'Accuracy':>10}")
print('-' * 47)

for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    preds = model.predict(X_test_tfidf)
    acc   = accuracy_score(y_test, preds)
    results[name] = {'model': model, 'preds': preds, 'accuracy': acc}
    print(f"{name:<35} {acc*100:>9.2f}%")

best_name = max(results, key=lambda k: results[k]['accuracy'])
print(f'\n🏆 Best model: {best_name} ({results[best_name]["accuracy"]*100:.2f}%)')

## 8. Detailed Evaluation of Best Model

In [ ]:
best = results[best_name]
print(f'=== Classification Report: {best_name} ===\n')
print(classification_report(y_test, best['preds'], target_names=['Fake', 'Real']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, best['preds'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Fake', 'Real'])
disp.plot(ax=axes[0], colorbar=False, cmap='RdYlGn')
axes[0].set_title(f'Confusion Matrix\n{best_name}', fontweight='bold')

# Model Accuracy Comparison
names = list(results.keys())
accs  = [results[n]['accuracy'] * 100 for n in names]
colors = ['#22c55e' if n == best_name else '#64748b' for n in names]
bars = axes[1].barh(names, accs, color=colors, edgecolor='white')
axes[1].set_xlim(0, 110)
axes[1].set_xlabel('Accuracy (%)')
axes[1].set_title('Model Comparison', fontweight='bold')
for bar, acc in zip(bars, accs):
    axes[1].text(acc + 0.5, bar.get_y() + bar.get_height()/2,
                 f'{acc:.1f}%', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Evaluation charts saved')

## 9. Top Features (Keywords) per Class

In [ ]:
# Works best with Logistic Regression
lr_model = results['Logistic Regression']['model']
feature_names = np.array(tfidf.get_feature_names_out())
coefs = lr_model.coef_[0]

top_n = 15
top_fake_idx = np.argsort(coefs)[:top_n]        # Most negative = Fake
top_real_idx = np.argsort(coefs)[-top_n:][::-1] # Most positive = Real

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, idx, title, color in [
    (axes[0], top_fake_idx, '🔴 Top Fake News Keywords', '#ef4444'),
    (axes[1], top_real_idx, '🟢 Top Real News Keywords', '#22c55e'),
]:
    words  = feature_names[idx]
    scores = np.abs(coefs[idx])
    ax.barh(words[::-1], scores[::-1], color=color, alpha=0.85)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Coefficient Magnitude')

plt.tight_layout()
plt.savefig('top_features.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Feature importance chart saved')

## 10. Predict on Custom Input

In [ ]:
def predict_news(text, model=None, vectorizer=None):
    """Predict whether a news article is Fake or Real."""
    if model is None:
        model = results[best_name]['model']
    if vectorizer is None:
        vectorizer = tfidf

    cleaned = preprocess_text(text)
    vec     = vectorizer.transform([cleaned])
    pred    = model.predict(vec)[0]

    # Probability (if model supports it)
    prob = None
    if hasattr(model, 'predict_proba'):
        prob = model.predict_proba(vec)[0]

    label = '✅ REAL NEWS' if pred == 1 else '🚨 FAKE NEWS'
    print('=' * 55)
    print(f'Input  : {text[:100]}...' if len(text) > 100 else f'Input  : {text}')
    print(f'Result : {label}')
    if prob is not None:
        print(f'Confidence → Fake: {prob[0]*100:.1f}%  |  Real: {prob[1]*100:.1f}%')
    print('=' * 55)
    return pred

# ── Test with sample articles ──────────────────────────────────────────────────
test_articles = [
    "SHOCKING: Government hiding alien contact for decades! Anonymous insider reveals truth that mainstream media won't tell you. Share before they delete this!",
    "The Federal Reserve raised interest rates by 25 basis points on Wednesday, citing continued concerns about inflation across major economic sectors.",
    "You won't BELIEVE what Big Pharma is hiding! One weird trick doctors refuse to share cures everything. They don't want you to know this secret!",
    "NASA scientists confirmed the successful launch of the Artemis III mission, which aims to return humans to the lunar surface by 2026."
]

print(f'Using model: {best_name}\n')
for article in test_articles:
    predict_news(article)
    print()

In [ ]:
# ── Predict YOUR OWN news article ─────────────────────────────────────────────
my_news = """
Paste or type your news article here and run this cell!
"""

predict_news(my_news.strip())

## 11. Save Best Model

In [ ]:
import joblib

joblib.dump(results[best_name]['model'], 'fake_news_model.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')

print('✅ Model saved  → fake_news_model.pkl')
print('✅ TF-IDF saved → tfidf_vectorizer.pkl')
print()
print('To load later:')
print('  model = joblib.load("fake_news_model.pkl")')
print('  tfidf = joblib.load("tfidf_vectorizer.pkl")')